# Formula & Document Conversion Workbench

**LaTeX ⇄ Unicode · MathML · DOCX/OMML · PDF · HTML · Markdown · PPTX · ODT**

A reusable Google Colab workbench for raising and lowering mathematical notation between representations and converting documents in batches.

**Copyright: © 2026 Abhishek Choudhary. All rights reserved.**

The original source is retained wherever possible. Unicode rendering is a presentation layer, not a guaranteed lossless substitute for every LaTeX macro, package, alignment environment, or custom command.

## 1. Install the toolchain
Run once per Colab runtime.

In [ ]:
import sys, subprocess, os, re, json, shutil, zipfile, tempfile
from pathlib import Path
from datetime import datetime, timezone

def shell(cmd):
    return subprocess.run(cmd, check=True, text=True, capture_output=True)

if shutil.which("pandoc") is None:
    shell(["bash", "-lc", "apt-get -qq update && apt-get -qq install -y pandoc texlive-xetex texlive-fonts-recommended poppler-utils"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "pylatexenc", "python-docx", "python-pptx", "pymupdf",
                       "lxml", "beautifulsoup4"])
print("Pandoc:", shutil.which("pandoc"), "| XeLaTeX:", shutil.which("xelatex"))

## 2. Configure copyright and workspace
Edit the copyright string here if required.

In [ ]:
COPYRIGHT = "© 2026 Abhishek Choudhary. All rights reserved."
AUTHOR = "Abhishek Choudhary"
WORKDIR = Path("/content/conversion_workbench")
INPUT_DIR, OUTPUT_DIR, TEMP_DIR = [WORKDIR / x for x in ("input", "output", "tmp")]
for folder in (INPUT_DIR, OUTPUT_DIR, TEMP_DIR):
    folder.mkdir(parents=True, exist_ok=True)
SUPPORTED_OUTPUTS = ["md", "tex", "html", "docx", "pptx", "odt", "pdf", "txt"]
print(COPYRIGHT)

## 3. LaTeX ⇄ Unicode

The initial renderer handles common Greek letters, operators, relations, fractions, roots, text wrappers, and simple scripts. The reverse converter handles common Unicode symbols and superscript/subscript runs.

Complex macros, matrices, cases, aligned equations, and package-specific syntax should remain in LaTeX or MathML. Treat raising as best-effort and inspect round-trip results.

In [ ]:
from pylatexenc.latex2text import LatexNodes2Text

GREEK = {
 r"\alpha":"α", r"\beta":"β", r"\gamma":"γ", r"\delta":"δ",
 r"\epsilon":"ε", r"\theta":"θ", r"\lambda":"λ", r"\mu":"μ",
 r"\pi":"π", r"\rho":"ρ", r"\sigma":"σ", r"\phi":"φ",
 r"\varphi":"ϕ", r"\omega":"ω", r"\Gamma":"Γ", r"\Delta":"Δ",
 r"\Theta":"Θ", r"\Lambda":"Λ", r"\Pi":"Π", r"\Sigma":"Σ",
 r"\Phi":"Φ", r"\Omega":"Ω"
}
SYMBOLS = {
 r"\times":"×", r"\cdot":"·", r"\div":"÷", r"\pm":"±",
 r"\leq":"≤", r"\le":"≤", r"\geq":"≥", r"\ge":"≥",
 r"\neq":"≠", r"\approx":"≈", r"\equiv":"≡", r"\infty":"∞",
 r"\partial":"∂", r"\nabla":"∇", r"\sum":"∑", r"\prod":"∏",
 r"\int":"∫", r"\sqrt":"√", r"\rightarrow":"→", r"\to":"→",
 r"\Rightarrow":"⇒", r"\leftarrow":"←", r"\forall":"∀",
 r"\exists":"∃", r"\in":"∈", r"\notin":"∉", r"\cup":"∪",
 r"\cap":"∩", r"\land":"∧", r"\lor":"∨", r"\therefore":"∴"
}
REV = {v:k for k,v in {**GREEK, **SYMBOLS}.items()}
REV.update({"−":"-", "⁄":r"\,/\,", "°":r"^\circ "})

SUP = dict(zip("⁰¹²³⁴⁵⁶⁷⁸⁹⁺⁻⁼⁽⁾ⁿⁱ", "0123456789+-=()ni"))
SUB = dict(zip("₀₁₂₃₄₅₆₇₈₉₊₋₌₍₎ₐₑₕᵢⱼₖₗₘₙₒₚᵣₛₜᵤᵥₓ",
               "0123456789+-=()ae hijkl mnoprstuvx".replace(" ","")))

def latex_to_unicode(latex):
    s = str(latex).strip()
    s = re.sub(r"^\$+|\$+$", "", s)
    s = re.sub(r"\\frac\s*\{([^{}]+)\}\s*\{([^{}]+)\}",
               lambda m: f"({m.group(1)})⁄({m.group(2)})", s)
    s = re.sub(r"\\sqrt\s*\{([^{}]+)\}", r"√(\1)", s)
    s = re.sub(r"\\(?:mathrm|mathbf|mathit|mathsf|mathtt|text)\s*\{([^{}]*)\}", r"\1", s)
    s = s.replace(r"\left", "").replace(r"\right", "")
    s = s.replace(r"\,", " ").replace(r"\;", " ").replace(r"\!", "")
    for k,v in sorted({**GREEK, **SYMBOLS}.items(), key=lambda kv: -len(kv[0])):
        s = s.replace(k, v)
    def scripts(m, table, marker):
        raw = m.group(1)
        return "".join(table[c] for c in raw) if all(c in table for c in raw) else marker + raw
    s = re.sub(r"\^\{([^{}]+)\}", lambda m: scripts(m, SUP, "^"), s)
    s = re.sub(r"_\{([^{}]+)\}", lambda m: scripts(m, SUB, "_"), s)
    try:
        s = LatexNodes2Text().latex_to_text(s)
    except Exception:
        pass
    return re.sub(r"\s+", " ", s).strip()

def unicode_to_latex(text):
    s = str(text)
    s = re.sub(r"\(([^()]*)\)⁄\(([^()]*)\)",
               lambda m: r"\frac{" + m.group(1) + "}{" + m.group(2) + "}", s)
    s = re.sub(r"[⁰¹²³⁴⁵⁶⁷⁸⁹⁺⁻⁼⁽⁾ⁿⁱ]+",
               lambda m: "^{" + "".join(SUP.get(c,c) for c in m.group()) + "}", s)
    s = re.sub(r"[₀₁₂₃₄₅₆₇₈₉₊₋₌₍₎ₐₑₕᵢⱼₖₗₘₙₒₚᵣₛₜᵤᵥₓ]+",
               lambda m: "_{" + "".join(SUB.get(c,c) for c in m.group()) + "}", s)
    for ch, macro in sorted(REV.items(), key=lambda kv: -len(kv[0])):
        s = s.replace(ch, macro)
    return s

def roundtrip(latex):
    u = latex_to_unicode(latex)
    return {"latex_input": latex, "unicode": u, "latex_raised": unicode_to_latex(u)}

tests = [
 r"L_{i,t}=f(K_{i,t},F_t,A_t,C_t,G_t)",
 r"K_{i,t+1}=(1-\delta)K_{i,t}+\varphi(P_{i,t},F_{i,t},E_{i,t})",
 r"\mathrm{ICER}=\frac{C_{\mathrm{AI}}-C_{\mathrm{C}}}{E_{\mathrm{AI}}-E_{\mathrm{C}}}",
 r"U_i=R_i+S_i+N_i-C_i"
]
for item in tests:
    print(roundtrip(item))

## 4. MathML and native Word equations

Pandoc can convert many standard LaTeX math expressions in Markdown into MathML for HTML and native Office Math Markup Language (OMML) equations for DOCX. Keep the original `.tex` as the canonical source, especially when custom packages or macros are used.

In [ ]:
def latex_to_mathml(fragment):
    proc = subprocess.run(["pandoc", "-f", "latex", "-t", "html"],
                          input=r"\(" + fragment + r"\)", text=True, capture_output=True)
    if proc.returncode:
        raise RuntimeError(proc.stderr)
    return proc.stdout

def markdown_to_docx(markdown, output_path):
    src = TEMP_DIR / "equation_source.md"
    src.write_text(markdown, encoding="utf-8")
    subprocess.run(["pandoc", str(src), "-o", str(output_path)], check=True)
    return output_path

print(latex_to_mathml(r"\frac{a}{b}+\alpha")[:700])

## 5. Extract text from common input formats

PDF extraction is text-based. Scanned PDFs need OCR. DOCX and PPTX extraction retains text but not all layout, images, charts, comments, or embedded objects.

In [ ]:
from docx import Document
from pptx import Presentation
import fitz
from bs4 import BeautifulSoup

def read_source(path):
    path = Path(path)
    ext = path.suffix.lower()
    if ext in (".md", ".txt", ".tex", ".html", ".htm"):
        data = path.read_text(encoding="utf-8", errors="replace")
        return BeautifulSoup(data, "html.parser").get_text("\n") if ext in (".html",".htm") else data
    if ext == ".docx":
        doc = Document(path)
        parts = [p.text for p in doc.paragraphs if p.text.strip()]
        for table in doc.tables:
            parts.extend(" | ".join(c.text for c in row.cells) for row in table.rows)
        return "\n\n".join(parts)
    if ext == ".pptx":
        prs = Presentation(path)
        parts = []
        for i, slide in enumerate(prs.slides, 1):
            parts.append(f"# Slide {i}")
            for shape in slide.shapes:
                if hasattr(shape, "text") and shape.text.strip():
                    parts.append(shape.text)
        return "\n\n".join(parts)
    if ext == ".pdf":
        return "\n\n".join(page.get_text() for page in fitz.open(path))
    raise ValueError("Unsupported input: " + ext)

## 6. Export to Markdown, LaTeX, HTML, DOCX, PPTX, ODT, PDF, or TXT

Copyright is added to generated text-oriented files and recorded in a sidecar manifest. It is not forcibly stamped over binary source documents.

In [ ]:
def add_notice(text, ext):
    ext = ext.lower().lstrip(".")
    if ext in ("md", "txt"):
        return text.rstrip() + "\n\n---\n\n" + COPYRIGHT + "\n"
    if ext == "tex":
        return "% " + COPYRIGHT + "\n\n" + text
    if ext == "html":
        return "<!-- " + COPYRIGHT + " -->\n" + text
    return text

def export_markdown(markdown, output_path, add_copyright=True):
    output_path = Path(output_path)
    ext = output_path.suffix.lower().lstrip(".")
    if ext not in SUPPORTED_OUTPUTS:
        raise ValueError("Unsupported output extension: " + ext)
    content = add_notice(markdown, ext) if add_copyright else markdown
    src = TEMP_DIR / ("source_" + str(os.getpid()) + ".md")
    src.write_text(content, encoding="utf-8")
    cmd = ["pandoc", str(src), "-o", str(output_path)]
    if ext == "pdf":
        cmd += ["--pdf-engine=xelatex", "-V", "mainfont=DejaVu Serif"]
    if ext == "html":
        cmd += ["--standalone", "--mathml"]
    if ext in ("tex", "pptx"):
        cmd += ["--standalone"]
    subprocess.run(cmd, check=True)
    manifest = {
        "source": src.name, "output": output_path.name, "format": ext,
        "author": AUTHOR, "copyright": COPYRIGHT,
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "engine": "Pandoc", "warning": "Inspect output; conversion may be lossy."
    }
    output_path.with_suffix(output_path.suffix + ".manifest.json").write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
    return output_path

demo = r"""# Formula test

$$
L_{i,t}=f(K_{i,t},F_t,A_t,C_t,G_t)
$$

$$
\mathrm{ICER}=\frac{C_{\mathrm{AI}}-C_{\mathrm{C}}}{E_{\mathrm{AI}}-E_{\mathrm{C}}}
$$
"""

## 7. Upload source files

Upload `.tex`, `.md`, `.txt`, `.html`, `.docx`, `.pptx`, or text-based `.pdf` files.

In [ ]:
from google.colab import files
uploaded = files.upload()
for filename, payload in uploaded.items():
    destination = INPUT_DIR / Path(filename).name
    destination.write_bytes(payload)
    print("Saved", destination)

## 8. Batch document conversion

Edit `OUTPUT_FORMATS` to choose the target formats. The intermediate text representation is convenient for broad conversion but will not preserve all original page design.

In [ ]:
OUTPUT_FORMATS = ["md", "tex", "html", "docx", "pptx", "pdf"]
conversion_log = []

for source in sorted(INPUT_DIR.iterdir()):
    if not source.is_file():
        continue
    try:
        extracted = read_source(source)
        for fmt in OUTPUT_FORMATS:
            target = OUTPUT_DIR / f"{source.stem}_converted.{fmt}"
            export_markdown(extracted, target)
            conversion_log.append({"source": source.name, "output": target.name, "status": "OK"})
    except Exception as exc:
        conversion_log.append({"source": source.name, "status": "ERROR", "error": str(exc)})

print(json.dumps(conversion_log, indent=2, ensure_ascii=False))

## 9. Batch formula conversion report

Edit the `FORMULAS` list or populate it from your own source parser. The CSV retains original LaTeX alongside Unicode output and raised LaTeX.

In [ ]:
import csv
FORMULAS = [
 r"L_{i,t}=f(K_{i,t},F_t,A_t,C_t,G_t)",
 r"K_{i,t+1}=(1-\delta)K_{i,t}+\varphi(P_{i,t},F_{i,t},E_{i,t})",
 r"\mathrm{ICER}=\frac{C_{\mathrm{AI}}-C_{\mathrm{C}}}{E_{\mathrm{AI}}-E_{\mathrm{C}}}",
 r"U_i=R_i+S_i+N_i-C_i",
 r"P_t=f(H_t,I_t,D_t)",
 r"X_{t+1}=(1-\Delta)X_t+J_t"
]
report = OUTPUT_DIR / "formula_roundtrip_report.csv"
with report.open("w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=["latex_input", "unicode", "latex_raised"])
    writer.writeheader()
    for formula in FORMULAS:
        writer.writerow(roundtrip(formula))
print("Created", report)

## 10. Package and download outputs

The ZIP includes converted files and their manifests.

In [ ]:
bundle = WORKDIR / "conversion_outputs_bundle.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as archive:
    for item in OUTPUT_DIR.rglob("*"):
        if item.is_file():
            archive.write(item, arcname=item.relative_to(OUTPUT_DIR))
print("Bundle:", bundle)
for item in sorted(OUTPUT_DIR.iterdir()):
    if item.is_file():
        print(item.name, item.stat().st_size, "bytes")

In [ ]:
from google.colab import files
files.download(str(bundle))

## 11. Validation checklist and known limits

- Compare each Unicode rendering with its source LaTeX.
- Manually inspect fractions, nested expressions, matrices, piecewise functions, aligned equations, and custom macros.
- Verify DOCX equations are editable OMML in Word or LibreOffice.
- Inspect PDF fonts, line breaks, and page breaks.
- Inspect PPTX slide layout and equation rendering.
- Keep original `.tex`, `.sty`, and `.cls` files with the project.
- A successful file conversion is not proof of mathematical equivalence.

**Limits:** Unicode raising is best-effort; PDF extraction is not OCR; document extraction can lose layout and embedded media. For publication PDFs, compile the original LaTeX project with its original preamble and dependencies.

**Copyright: © 2026 Abhishek Choudhary. All rights reserved.**